# TBA Total Return Swaps in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Portfolio and transactions |
| 4 | Valuation |
| 5 | The swap's maturity as an event |

## The instrument

A TBA total return swap has two legs pulling in opposite directions: one side carries the return
on a to-be-announced agency pass-through (the asset leg), and the other pays a fixed financing
cost (the funding leg):

    asset leg     the TBA, as a mastered ToBeAnnounced referenced by ReferenceInstrument
    funding leg   a FixedLeg paying a fixed financing rate

A `ToBeAnnounced` represents a generic agency MBS forward in LUSID -- just coupon, tenor and
agency, with no pool-specific CUSIP, coupon cashflows, accrual or factor attached. You master it
once like any other instrument, and the swap's asset leg simply points at it with a
`ReferenceInstrument`.

## Date constraint on the referenced TBA

**The TBA you reference has to sandwich the swap's own dates: its `start_date` on or before the
swap's `start_date`, and its `maturity_date` on or after the swap's `maturity_date`, exactly.**
LUSID's date validation rejects the upsert if that's not true.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

`SimpleStatic` just prices the position off the quoted mark.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "TbaTrsDemo"
RECIPE    = "tba-trs-demo-recipe"
PORTFOLIO = "tba-trs-demo-book"

TBA_ID    = "DEMO-TBA-FNMA-30YR-01"
SWAP_ID   = "DEMO-TBA-TRS-01"
DESC      = "Demo FNMA 5.5% 30YR TBA TRS"
CURRENCY  = "USD"
START     = d(2025, 6, 1)
MATURITY  = d(2026, 6, 1)
ASOF      = d(2025, 12, 1)

AGENCY    = "FNMA"
COUPON    = 5.5
TENOR     = "30YR"

FUNDING_RATE       = 0.0400          # fixed financing rate paid on the funding leg
FUNDING_FREQUENCY  = "3M"
FUNDING_DAY_COUNT  = "Actual360"
ASSET_SIDE   = "Receive"
FUNDING_SIDE = "Pay"

NOTIONAL        = 1.0                # unit notional makes this a per-unit contract
INITIAL_PRICE   = 1.0
RESET_FREQUENCY = "3M"               # reset frequency for the asset leg's reset schedule

QUANTITY = 5_000_000.00              # face value referenced
PRICE    = 1.25                      # points, quoted mark
DENOM    = 100

print(f"{DESC}")
print(f"  asset   {ASSET_SIDE:<8} {TBA_ID} ({AGENCY} {COUPON}% {TENOR})")
print(f"  funding {FUNDING_SIDE:<8} fixed {FUNDING_RATE:.2%} ({FUNDING_FREQUENCY} {FUNDING_DAY_COUNT})")
print(f"  {QUANTITY:,.0f} face at {PRICE} points on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE} / {DENOM} = {QUANTITY * PRICE / DENOM:,.2f} {CURRENCY}")

Demo FNMA 5.5% 30YR TBA TRS
  asset   Receive  DEMO-TBA-FNMA-30YR-01 (FNMA 5.5% 30YR)
  funding Pay      fixed 4.00% (3M Actual360)
  5,000,000 face at 1.25 points on 2025-12-01
  market value = 5,000,000 x 1.25 / 100 = 62,500.00 USD


---
# 1. Instrument creation

## The underlying TBA

Agency TBAs trade under a generic CUSIP rather than a pool-specific one -- `ToBeAnnounced` reflects
that by taking agency, coupon and tenor directly, with no CUSIP or pool identifier at all. We
master it once, and date it to sandwich the swap.

In [3]:
tba = m.ToBeAnnounced(
    instrument_type="ToBeAnnounced",
    start_date=START,
    maturity_date=MATURITY,
    dom_ccy=CURRENCY,
    agency=AGENCY,
    coupon=COUPON,
    tenor=TENOR,
    trading_conventions=m.TradingConventions(price_scale_factor=DENOM))

TBA_LUID = upsert("tba", f"{AGENCY} TBA {COUPON}% {TENOR}", TBA_ID, tba)
print(f"TBA : {TBA_LUID}")

TBA : LUID_00003DG4


## The swap

The asset leg points at the mastered TBA through a `ReferenceInstrument`. The funding leg is just
a plain `FixedLeg` -- no fixing or rate quotes to worry about.

In [4]:
trs = m.TotalReturnSwap(
    instrument_type="TotalReturnSwap",
    start_date=START,
    maturity_date=MATURITY,
    asset_leg=m.AssetLeg(
        asset=m.ReferenceInstrument(
            instrument_type="ReferenceInstrument",
            instrument_id=TBA_LUID,
            instrument_id_type="LusidInstrumentId",
            scope=SCOPE),
        pay_receive=ASSET_SIDE,
        initial_price=INITIAL_PRICE,
        reset_schedule=m.ResetSchedule(frequency=RESET_FREQUENCY),
        income_policy="PassThrough"),
    funding_leg=m.FixedLeg(
        instrument_type="FixedLeg",
        start_date=START,
        maturity_date=MATURITY,
        notional=NOTIONAL,
        leg_definition=m.LegDefinition(
            rate_or_spread=FUNDING_RATE,
            pay_receive=FUNDING_SIDE,
            conventions=m.FlowConventions(
                currency=CURRENCY,
                payment_frequency=FUNDING_FREQUENCY,
                day_count_convention=FUNDING_DAY_COUNT,
                roll_convention=str(START.day),
                payment_calendars=[], reset_calendars=[]),
            stub_type="ShortBack",
            notional_exchange_type="None")))

TRS_LUID = upsert("trs", DESC, SWAP_ID, trs)
print(f"TBA TRS : {TRS_LUID}")

TBA TRS : LUID_00003DG5


---
# 2. Recipe

Under `SimpleStatic`, the position gets reported at its quoted mark. The legs' conventions
describe what the instrument *is* -- under this model, they don't feed into the valuation number
itself.

In [5]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="TBA TRS, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="TotalReturnSwap")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: TbaTrsDemo/tba-trs-demo-recipe


---
# 3. Portfolio and transactions

No principal changes hands when you strike a TRS, so `totalConsideration` comes in at zero.

In [6]:
recreate_portfolio(PORTFOLIO, "TBA TRS Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-TBA-TRS",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": TRS_LUID},
        transaction_date=START.isoformat(),
        settlement_date=START.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, START, START))

Recreated TbaTrsDemo/tba-trs-demo-book


,date,type,luid,units,consideration
0,2025-06-01,Buy,LUID_00003DG5,"5,000,000.00",0.00


---
# 4. Valuation

We quote the swap's own price, per unit. A `ToBeAnnounced` always gets valued off its own EOD
quote lookup, no matter what the recipe's model rules say, so the referenced TBA needs a quote
too -- any level works here, since it's the swap's own quote that drives `CleanPV` under
`SimpleStatic`.

In [7]:
upsert_price(TBA_LUID, PRICE / DENOM, ASOF, CURRENCY)
upsert_price(TRS_LUID, PRICE / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV)
0,Demo FNMA 5.5% 30YR TBA TRS,"5,000,000.00","62,500.00"


LUSID CleanPV 62,500.00  vs  quoted mark 62,500.00


---
# 5. The swap's maturity as an event

Every `TotalReturnSwap` carries a `maturity_date`, and LUSID forecasts what happens on that date
as a `MaturityEvent` -- you don't need to declare any extra schedule on the instrument itself.
What LUSID does need is a portfolio that's configured to forecast events: `instrumentEventConfiguration`
has to point at a recipe, and that can only be set when the portfolio is created. Passing
`recipe=RECIPE` into `recreate_portfolio()` back in Section 3 took care of that. Skip it, and
`query_applicable_instrument_events` just comes back empty rather than throwing an error.

Query a window that runs past `MATURITY` and you'll see the `MaturityEvent` itself, forecast as an
actual event on the position -- not just inferred from the instrument's own dates.

In [8]:
events_api = api(lusid.InstrumentEventsApi)

WINDOW_END = MATURITY + timedelta(days=5)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=ASOF.isoformat(),
        window_end=WINDOW_END.isoformat(),
        effective_at=WINDOW_END.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

,event type,eligible balance,status
0,MaturityEvent,"5,000,000.00",Active


---
# Summary

1. A TBA TRS is a `TotalReturnSwap` whose asset leg points at a mastered `ToBeAnnounced` through a
   `ReferenceInstrument` -- the TBA itself only carries agency, coupon and tenor, with no
   pool-specific data.
2. The TBA's `start_date`/`maturity_date` has to sandwich the swap's own dates, so this notebook
   just dates the TBA to match the swap's dates directly.
3. The funding leg is a plain `FixedLeg` -- no fixing or rate quotes required.
4. Valuation runs off a quoted mark under `SimpleStatic` -- the legs describe what the instrument
   is, not what it's worth.
5. The swap's `MaturityEvent` becomes forecastable once the portfolio's
   `instrumentEventConfiguration` points at a recipe -- set at creation via `recreate_portfolio(...,
   recipe=RECIPE)` -- and needs no extra schedule on the `TotalReturnSwap` itself.

In [9]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"TBA        : {TBA_LUID}")
print(f"TBA TRS    : {TRS_LUID}")

Scope      : TbaTrsDemo
Portfolio  : TbaTrsDemo/tba-trs-demo-book
Recipe     : TbaTrsDemo/tba-trs-demo-recipe
TBA        : LUID_00003DG4
TBA TRS    : LUID_00003DG5
